# Ensemble Stacking Model — LSTM + GRU + TFT

**Auto-selecting stacking ensemble: trains 6 meta-learners, picks the best by val MSE**

| Base Model | Window | Config | Checkpoint |
|:-----------|:------:|:------:|:-----------|
| LSTM       |   30   | small (hidden=32) | `w30_small.pt` |
| GRU        |   30   | small (hidden=32) | `w30_small.pt` |
| TFT        |   15   | hidden=32, attn=4 | `best-epoch003-valloss0.007141.ckpt` |

- Each base model outputs **7 quantiles** → **21 stacked features**
- 6 meta-learners: **Ridge, ElasticNet, GBR, RF, SVR, MLP** — all trained on same val split
- Auto-selects winner by **val MSE** (no test leakage)
- All base model weights are **frozen**; only the meta-learner is trained
- Target: `target_pct_change` (next-day return fraction)

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
%%capture
!pip install scikit-learn matplotlib seaborn tqdm pytorch-forecasting lightning

In [46]:
import os, sys, json, math, random, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score
)

import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB
PyTorch: 2.10.0+cu128


In [47]:
# ── Paths to pre-trained model weights ────────────────────────────────
GRU_CKPT  = Path('/content/drive/MyDrive/inlp/GRU_outputs/gru_outputs/checkpoints/w30_small.pt')
LSTM_CKPT = Path('/content/drive/MyDrive/inlp/lstm_outputs/lstm_outputs-2/checkpoints/w30_small.pt')
TFT_CKPT  = Path('/content/drive/MyDrive/inlp/window_15/checkpoints/best-epoch003-valloss0.007200.ckpt')

# Kaggle fallbacks
if not GRU_CKPT.exists():
    GRU_CKPT = Path('/kaggle/input/model-weights/gru_outputs/checkpoints/w30_small.pt')
if not LSTM_CKPT.exists():
    LSTM_CKPT = Path('/kaggle/input/model-weights/lstm_outputs-2/checkpoints/w30_small.pt')
if not TFT_CKPT.exists():
    TFT_CKPT = Path('/kaggle/input/model-weights/tft/window_15/checkpoints/best-epoch003-valloss0.007141.ckpt')

DATA_PATH = Path('/content/drive/MyDrive/inlp/dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/kaggle/input/dataset/dataset.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('dataset.csv not found!')

LSTM_WINDOW = 30
GRU_WINDOW  = 30
TFT_WINDOW  = 15

TRAIN_END  = pd.Timestamp('2024-12-31')
VAL_START  = pd.Timestamp('2025-01-01')
VAL_END    = pd.Timestamp('2025-06-30')
TEST_START = pd.Timestamp('2025-07-01')

QUANTILES  = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]
MEDIAN_IDX = 3
NUM_QUANTILES = 7

TARGET_COL   = 'target_pct_change'
PRICE_COL    = 'Adj Close'
DATE_COL     = 'Date'
SYMBOL_COL   = 'Symbol'
TIME_IDX_COL = 'time_idx'
EXCLUDE_COLS = {'Date','Symbol','Open','High','Low','Close','Adj Close','Volume','symbol_base','time_idx'}
EPS = 1e-8
NUM_WORKERS = 0
META_EPOCHS = 100
META_LR = 1e-3
META_BS = 512
META_PATIENCE = 15

OUTPUT_DIR = Path('ensemble_outputs')
for _sub in ['checkpoints', 'plots', 'results']:
    (OUTPUT_DIR / _sub).mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
for _p, _n in [(GRU_CKPT, 'GRU'), (LSTM_CKPT, 'LSTM'), (TFT_CKPT, 'TFT')]:
    print(f'  {_n}: {_p} | exists={_p.exists()}')
print(f'  Data: {DATA_PATH}')

Configuration loaded.
  GRU: /content/drive/MyDrive/inlp/GRU_outputs/gru_outputs/checkpoints/w30_small.pt | exists=True
  LSTM: /content/drive/MyDrive/inlp/lstm_outputs/lstm_outputs-2/checkpoints/w30_small.pt | exists=True
  TFT: /content/drive/MyDrive/inlp/window_15/checkpoints/best-epoch003-valloss0.007200.ckpt | exists=True
  Data: /content/drive/MyDrive/inlp/dataset.csv


## 1. Data Loading & Feature Engineering

In [48]:
def load_and_prepare(path: Path):
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = df.dropna(subset=[DATE_COL])
    df = df.sort_values([SYMBOL_COL, DATE_COL]).reset_index(drop=True)
    df = df.drop_duplicates([SYMBOL_COL, DATE_COL])
    df['dow']            = df[DATE_COL].dt.weekday.astype(np.float32)
    df['dom']            = df[DATE_COL].dt.day.astype(np.float32)
    df['month']          = df[DATE_COL].dt.month.astype(np.float32)
    df['is_month_start'] = df[DATE_COL].dt.is_month_start.astype(np.float32)
    df['is_month_end']   = df[DATE_COL].dt.is_month_end.astype(np.float32)
    num_cols  = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    feat_cols = [c for c in num_cols if c not in EXCLUDE_COLS]
    if TARGET_COL in feat_cols:
        feat_cols = [TARGET_COL] + [c for c in feat_cols if c != TARGET_COL]
    df[feat_cols] = df[feat_cols].fillna(0.0)
    print(f'Loaded: {df.shape[0]:,} rows | {df[SYMBOL_COL].nunique()} symbols')
    print(f'Dates : {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}')
    print(f'Features ({len(feat_cols)}): {feat_cols[:5]} ...')
    return df, feat_cols

df_raw, FEATURE_COLS = load_and_prepare(DATA_PATH)
N_FEATURES = len(FEATURE_COLS)
SYMBOLS = sorted(df_raw[SYMBOL_COL].unique())
print(f'Symbols: {len(SYMBOLS)}')

Loaded: 71,938 rows | 50 symbols
Dates : 2020-03-12 to 2026-03-27
Features (34): ['target_pct_change', 'adj_ret_1d', 'price_range_pct', 'gap_pct', 'rolling_volatility_5d'] ...
Symbols: 50


## 2. Base Model Architectures (GRU & LSTM)

In [49]:
class GRUQuantile(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, num_quantiles=7):
        super().__init__()
        self.hidden_size = hidden_size; self.num_layers = num_layers; self.num_quantiles = num_quantiles
        self.gru1 = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.act1 = nn.Tanh(); self.drop1 = nn.Dropout(dropout)
        self.gru2 = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act2 = nn.Tanh(); self.drop2 = nn.Dropout(dropout)
        self.gru3 = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act3 = nn.Tanh(); self.drop3 = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_size)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.gelu = nn.GELU(); self.drop4 = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size // 2, num_quantiles)
    def forward(self, x):
        out, _ = self.gru1(x); out = self.drop1(self.act1(out)); res = out
        out, _ = self.gru2(out); out = self.drop2(self.act2(out)) + res; res = out
        out, _ = self.gru3(out); out = self.drop3(self.act3(out)) + res
        last = self.norm(out[:, -1, :]); h = self.drop4(self.gelu(self.fc1(last)))
        return self.head(h)

class LSTMQuantile(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, num_quantiles=7):
        super().__init__()
        self.hidden_size = hidden_size; self.num_layers = num_layers; self.num_quantiles = num_quantiles
        self.lstm1 = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.act1 = nn.Tanh(); self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act2 = nn.Tanh(); self.drop2 = nn.Dropout(dropout)
        self.lstm3 = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act3 = nn.Tanh(); self.drop3 = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_size)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.gelu = nn.GELU(); self.drop4 = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size // 2, num_quantiles)
    def forward(self, x):
        out, _ = self.lstm1(x); out = self.drop1(self.act1(out)); res = out
        out, _ = self.lstm2(out); out = self.drop2(self.act2(out)) + res; res = out
        out, _ = self.lstm3(out); out = self.drop3(self.act3(out)) + res
        last = self.norm(out[:, -1, :]); h = self.drop4(self.gelu(self.fc1(last)))
        return self.head(h)

print('GRU & LSTM architectures defined.')

GRU & LSTM architectures defined.


## 3. Sliding-Window Dataset (LSTM / GRU)

In [50]:
class StockWindowDataset(Dataset):
    def __init__(self, df, feature_cols, window_size, split, scaler=None, fit_scaler=False):
        assert split in ('train', 'val', 'test')
        self.window_size = window_size; self.feature_cols = feature_cols
        xs, ys, bpxs, tpxs = [], [], [], []
        syms_list, date_list, train_feats_list = [], [], []
        for sym, grp in df.groupby(SYMBOL_COL, sort=True):
            grp = grp.sort_values(DATE_COL).reset_index(drop=True)
            if len(grp) < window_size + 1: continue
            feats = np.nan_to_num(grp[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            targets = grp[TARGET_COL].values.astype(np.float32)
            prices = grp[PRICE_COL].values.astype(np.float32)
            dt_arr = grp[DATE_COL].values
            if fit_scaler:
                tmask = pd.DatetimeIndex(dt_arr) <= TRAIN_END
                if tmask.any(): train_feats_list.append(feats[tmask])
            for t in range(window_size, len(grp)):
                d = pd.Timestamp(dt_arr[t])
                in_split = {'train': d<=TRAIN_END, 'val': VAL_START<=d<=VAL_END, 'test': d>=TEST_START}[split]
                if not in_split: continue
                xs.append(feats[t-window_size:t]); ys.append(targets[t])
                bpxs.append(prices[t-1]); tpxs.append(prices[t])
                syms_list.append(sym); date_list.append(d)
        if fit_scaler and train_feats_list:
            scaler = StandardScaler(); scaler.fit(np.vstack(train_feats_list))
        self.scaler = scaler
        if len(xs) > 0:
            xs_arr = np.stack(xs).astype(np.float32)
            if self.scaler is not None:
                N,W,F = xs_arr.shape; xs_arr = self.scaler.transform(xs_arr.reshape(-1,F)).reshape(N,W,F)
            self.xs = xs_arr
        else: self.xs = np.zeros((0, window_size, len(feature_cols)), np.float32)
        self.ys = np.array(ys, np.float32); self.base_pxs = np.array(bpxs, np.float32)
        self.true_pxs = np.array(tpxs, np.float32)
        self.symbols = np.array(syms_list, object); self.dates = np.array(date_list, object)
        n_sym = len(np.unique(self.symbols)) if len(self.symbols)>0 else 0
        print(f'  [{split:5s}] {len(self.xs):7,} samples | {n_sym:3d} symbols | window={window_size}')
    def __len__(self): return len(self.xs)
    def __getitem__(self, idx):
        return (torch.from_numpy(self.xs[idx]), torch.tensor(self.ys[idx], dtype=torch.float32),
                torch.tensor(self.base_pxs[idx], dtype=torch.float32), torch.tensor(self.true_pxs[idx], dtype=torch.float32))

def build_datasets(df, feature_cols, window_size):
    print(f'Building datasets (window={window_size}) ...')
    tr = StockWindowDataset(df, feature_cols, window_size, 'train', fit_scaler=True)
    va = StockWindowDataset(df, feature_cols, window_size, 'val', scaler=tr.scaler)
    te = StockWindowDataset(df, feature_cols, window_size, 'test', scaler=tr.scaler)
    return tr, va, te, tr.scaler
print('Dataset class defined.')

Dataset class defined.


In [51]:
train30, val30, test30, scaler30 = build_datasets(df_raw, FEATURE_COLS, 30)

Building datasets (window=30) ...
  [train]  55,316 samples |  49 symbols | window=30
  [val  ]   6,027 samples |  49 symbols | window=30
  [test ]   9,095 samples |  50 symbols | window=30


## 4. Metrics & Inference Helpers

In [52]:
def compute_metrics(y_true, y_pred, base_px, true_px):
    valid = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(base_px) & np.isfinite(true_px)
    yt, yp, bp, tp = y_true[valid], y_pred[valid], base_px[valid], true_px[valid]
    if len(yt)==0: return {'n_samples':0}
    pred_px = bp*(1.0+yp); denom_px = np.where(np.abs(tp)<EPS,EPS,np.abs(tp))
    mape_px = float(np.mean(np.abs(pred_px-tp)/denom_px)*100.0)
    err_pct = (yp-yt)*100.0
    mse_p=float(np.mean(err_pct**2)); rmse_p=float(np.sqrt(mse_p)); mae_p=float(np.mean(np.abs(err_pct)))
    true_up=(yt>0).astype(int); pred_up=(yp>0).astype(int); dir_acc=float(np.mean(true_up==pred_up))
    return {'MAPE_price':mape_px,'MSE_pct':mse_p,'RMSE_pct':rmse_p,'MAE_pct':mae_p,
            'F1_macro':float(f1_score(true_up,pred_up,average='macro',zero_division=0)),
            'Accuracy':float(accuracy_score(true_up,pred_up)),
            'Precision_macro':float(precision_score(true_up,pred_up,average='macro',zero_division=0)),
            'Recall_macro':float(recall_score(true_up,pred_up,average='macro',zero_division=0)),
            'Directional_Acc':dir_acc,'n_samples':len(yt),
            'pct_up_true':float(np.mean(true_up)),'pct_up_pred':float(np.mean(pred_up))}

REPORT_COLS = ['MAPE_price','MSE_pct','RMSE_pct','MAE_pct','F1_macro','Accuracy',
               'Precision_macro','Recall_macro','Directional_Acc','n_samples']

@torch.no_grad()
def get_all_quantile_preds(model, dataset, batch_size=512):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    model.eval()
    all_preds, all_trues, all_bp, all_tp = [], [], [], []
    for x, y, bp, tp in loader:
        out = model(x.to(DEVICE))
        all_preds.append(out.cpu().numpy()); all_trues.append(y.numpy())
        all_bp.append(bp.numpy()); all_tp.append(tp.numpy())
    return {'preds_q':np.concatenate(all_preds),'trues':np.concatenate(all_trues),
            'base_px':np.concatenate(all_bp),'true_px':np.concatenate(all_tp),
            'symbols':dataset.symbols,'dates':dataset.dates}
print('Metrics & inference helpers defined.')

Metrics & inference helpers defined.


## 5. Load & Freeze GRU and LSTM Base Models

In [53]:
gru = GRUQuantile(input_size=N_FEATURES, hidden_size=32, num_layers=1, dropout=0.10, num_quantiles=7).to(DEVICE)
gru.load_state_dict(torch.load(GRU_CKPT, map_location=DEVICE))
for p in gru.parameters(): p.requires_grad = False
gru.eval()
print(f'GRU  loaded & frozen  ({sum(p.numel() for p in gru.parameters()):,} params)')

lstm = LSTMQuantile(input_size=N_FEATURES, hidden_size=32, num_layers=1, dropout=0.10, num_quantiles=7).to(DEVICE)
lstm.load_state_dict(torch.load(LSTM_CKPT, map_location=DEVICE))
for p in lstm.parameters(): p.requires_grad = False
lstm.eval()
print(f'LSTM loaded & frozen  ({sum(p.numel() for p in lstm.parameters()):,} params)')

print('\nGRU val predictions ...');  gru_val  = get_all_quantile_preds(gru, val30)
print('GRU test predictions ...'); gru_test = get_all_quantile_preds(gru, test30)
print('\nLSTM val predictions ...');  lstm_val  = get_all_quantile_preds(lstm, val30)
print('LSTM test predictions ...'); lstm_test = get_all_quantile_preds(lstm, test30)
print(f'\nGRU  val={gru_val["trues"].shape[0]:,}  test={gru_test["trues"].shape[0]:,}')
print(f'LSTM val={lstm_val["trues"].shape[0]:,}  test={lstm_test["trues"].shape[0]:,}')

GRU  loaded & frozen  (19,911 params)
LSTM loaded & frozen  (26,311 params)

GRU val predictions ...
GRU test predictions ...

LSTM val predictions ...
LSTM test predictions ...

GRU  val=6,027  test=9,095
LSTM val=6,027  test=9,095


## 6. TFT Data Pipeline & Frozen Inference

In [54]:
RAW_FEATURE_DROP = {'Open','High','Low','Close','Adj Close','Volume','symbol_base',DATE_COL}
TFT_KNOWN_CATS = ['dow','dom','month','is_month_start','is_month_end']

def prepare_tft_df(path):
    df = pd.read_csv(path); df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = df.dropna(subset=[DATE_COL]).sort_values([SYMBOL_COL,DATE_COL]).reset_index(drop=True)
    df = df.drop_duplicates([SYMBOL_COL,DATE_COL]); df[TIME_IDX_COL] = df[TIME_IDX_COL].astype(np.int64)
    df['dow']=df[DATE_COL].dt.weekday.astype(np.int16).astype(str)
    df['dom']=df[DATE_COL].dt.day.astype(np.int16).astype(str)
    df['month']=df[DATE_COL].dt.month.astype(np.int16).astype(str)
    df['is_month_start']=df[DATE_COL].dt.is_month_start.astype(np.int8).astype(str)
    df['is_month_end']=df[DATE_COL].dt.is_month_end.astype(np.int8).astype(str)
    return df

def build_tft_datasets(df, enc_len=15, pred_len=1):
    train_mask = df[DATE_COL]<=pd.Timestamp(TRAIN_END)
    train_counts = df.loc[train_mask].groupby(SYMBOL_COL).size()
    eligible = [s for s in sorted(df[SYMBOL_COL].unique()) if train_counts.get(s,0)>=enc_len+pred_len+1]
    work = df[df[SYMBOL_COL].isin(eligible)].copy()
    known_cats = [c for c in TFT_KNOWN_CATS if c in work.columns]; known_reals = [TIME_IDX_COL]
    numeric = [c for c in work.columns if pd.api.types.is_numeric_dtype(work[c])]
    excl = set(known_reals+[TARGET_COL,TIME_IDX_COL])
    unk_reals = [TARGET_COL]+[c for c in numeric if c not in excl and c not in RAW_FEATURE_DROP]
    seen=set(); unk_reals=[c for c in unk_reals if not(c in seen or seen.add(c))]
    train_df=work.loc[work[DATE_COL]<=pd.Timestamp(TRAIN_END)].copy()
    val_df=work.loc[work[DATE_COL]<=pd.Timestamp(VAL_END)].copy(); test_df=work.copy()
    drop=[c for c in RAW_FEATURE_DROP if c in train_df.columns and c!=DATE_COL]
    train_m=train_df.drop(columns=drop,errors='ignore'); val_m=val_df.drop(columns=drop,errors='ignore')
    test_m=test_df.drop(columns=drop,errors='ignore')
    training = TimeSeriesDataSet(train_m,time_idx=TIME_IDX_COL,target=TARGET_COL,group_ids=[SYMBOL_COL],
        min_encoder_length=enc_len,max_encoder_length=enc_len,min_prediction_length=pred_len,max_prediction_length=pred_len,
        static_categoricals=[SYMBOL_COL],time_varying_known_categoricals=known_cats,
        time_varying_known_reals=known_reals,time_varying_unknown_reals=unk_reals,
        target_normalizer=GroupNormalizer(groups=[SYMBOL_COL],method='standard'),
        add_relative_time_idx=False,add_target_scales=True,add_encoder_length=True,allow_missing_timesteps=True)
    vs_idx=int(work.loc[work[DATE_COL]>=pd.Timestamp(VAL_START),TIME_IDX_COL].min())
    ts_idx=int(work.loc[work[DATE_COL]>=pd.Timestamp(TEST_START),TIME_IDX_COL].min())
    val_ds=TimeSeriesDataSet.from_dataset(training,val_m,min_prediction_idx=vs_idx,stop_randomization=True)
    test_ds=TimeSeriesDataSet.from_dataset(training,test_m,min_prediction_idx=ts_idx,stop_randomization=True)
    return training, val_ds, test_ds, work

print('Preparing TFT data ...')
tft_df = prepare_tft_df(DATA_PATH)
tft_train_ds, tft_val_ds, tft_test_ds, tft_work = build_tft_datasets(tft_df, enc_len=TFT_WINDOW)
print(f'TFT val samples : {len(tft_val_ds):,}')
print(f'TFT test samples: {len(tft_test_ds):,}')

Preparing TFT data ...
TFT val samples : 6,027
TFT test samples: 9,061


In [55]:
# Load TFT from checkpoint
import torch.serialization, pandas.core.internals.managers, pandas._libs.internals
torch.serialization.add_safe_globals([GroupNormalizer,pd.DataFrame,pandas.core.internals.managers.BlockManager,
    pandas._libs.internals._unpickle_block,np.ndarray,np.dtype,np.float64])
checkpoint = torch.load(TFT_CKPT, map_location='cpu', weights_only=False)
hparams_from_ckpt = checkpoint['hyper_parameters']
tft_model = TemporalFusionTransformer(**hparams_from_ckpt)
state_dict = checkpoint['state_dict']
new_state_dict = {}
for k, v in state_dict.items():
    new_state_dict[k[6:] if k.startswith('model.') else k] = v
tft_model.load_state_dict(new_state_dict)
for p in tft_model.parameters(): p.requires_grad = False
tft_model.eval()
print(f'TFT  loaded & frozen  ({sum(p.numel() for p in tft_model.parameters()):,} params)')

@torch.no_grad()
def get_tft_predictions(model, dataset, work_df, batch_size=64):
    loader = dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=NUM_WORKERS)
    pred_output = model.predict(loader, mode='prediction', return_index=True,
        trainer_kwargs={'logger':False,'enable_checkpointing':False})
    if hasattr(pred_output,'output'): preds=pred_output.output; idx_df=pred_output.index.copy()
    elif isinstance(pred_output,tuple):
        preds,idx_df=None,None
        for item in pred_output:
            if isinstance(item,pd.DataFrame): idx_df=item.copy()
            elif torch.is_tensor(item) or isinstance(item,np.ndarray): preds=item
            elif hasattr(item,'output'): preds=item.output; idx_df=getattr(item,'index',idx_df)
    else: preds=pred_output; idx_df=None
    if torch.is_tensor(preds): preds=preds.cpu().numpy()
    if preds.ndim==3: preds_q=preds[:,0,:]
    elif preds.ndim==2: preds_q=preds
    else: preds_q=preds[:,None]
    sym_col=SYMBOL_COL
    for c in [SYMBOL_COL,'__group_id__Symbol']:
        if c in idx_df.columns: sym_col=c; break
    time_col=TIME_IDX_COL
    for c in [TIME_IDX_COL,'decoder_time_idx','__time_idx__']:
        if c in idx_df.columns: time_col=c; break
    n=len(preds_q); symbols=idx_df[sym_col].astype(str).values[:n]; time_idxs=idx_df[time_col].astype(np.int64).values[:n]
    lk_tgt=work_df.set_index([SYMBOL_COL,TIME_IDX_COL])[TARGET_COL].to_dict()
    lk_px=work_df.set_index([SYMBOL_COL,TIME_IDX_COL])[PRICE_COL].to_dict()
    lk_dt=work_df.drop_duplicates(TIME_IDX_COL).set_index(TIME_IDX_COL)[DATE_COL].to_dict()
    trues=np.array([lk_tgt.get((s,int(t)),np.nan) for s,t in zip(symbols,time_idxs)],dtype=np.float32)
    base_px=np.array([lk_px.get((s,int(t)-1),np.nan) for s,t in zip(symbols,time_idxs)],dtype=np.float32)
    true_px=np.array([lk_px.get((s,int(t)),np.nan) for s,t in zip(symbols,time_idxs)],dtype=np.float32)
    dates=np.array([lk_dt.get(int(t),pd.NaT) for t in time_idxs])
    return {'preds_q':preds_q.astype(np.float32),'trues':trues,'base_px':base_px,'true_px':true_px,'symbols':symbols,'dates':dates}

print('\nTFT val predictions ...')
tft_val = get_tft_predictions(tft_model, tft_val_ds, tft_work)
print(f'  Val samples: {tft_val["trues"].shape[0]:,}')
print('TFT test predictions ...')
tft_test = get_tft_predictions(tft_model, tft_test_ds, tft_work)
print(f'  Test samples: {tft_test["trues"].shape[0]:,}')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


TFT  loaded & frozen  (117,915 params)

TFT val predictions ...


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Val samples: 6,027
TFT test predictions ...
  Test samples: 9,061


## 7. Align Predictions Across Models

In [56]:
def build_pred_df(pred_dict, model_name):
    n = len(pred_dict['trues'])
    df = pd.DataFrame({'symbol':pred_dict['symbols'][:n],'date':pd.to_datetime(pred_dict['dates'][:n]),
        'true':pred_dict['trues'][:n],'base_px':pred_dict['base_px'][:n],'true_px':pred_dict['true_px'][:n]})
    preds_q = pred_dict['preds_q'][:n]
    if preds_q.ndim == 1: preds_q = preds_q[:, None]
    n_q = preds_q.shape[1]
    if n_q == 1:
        for qi in range(NUM_QUANTILES): df[f'{model_name}_q{qi}'] = preds_q[:, 0]
        df[f'{model_name}_median'] = preds_q[:, 0]
    else:
        for qi in range(n_q): df[f'{model_name}_q{qi}'] = preds_q[:, qi]
        df[f'{model_name}_median'] = preds_q[:, min(MEDIAN_IDX, n_q-1)]
    return df

def align_predictions(lstm_p, gru_p, tft_p):
    df_lstm=build_pred_df(lstm_p,'lstm'); df_gru=build_pred_df(gru_p,'gru'); df_tft=build_pred_df(tft_p,'tft')
    merged = df_lstm.merge(df_gru.drop(columns=['true','base_px','true_px']),on=['symbol','date'],how='inner'
        ).merge(df_tft.drop(columns=['true','base_px','true_px']),on=['symbol','date'],how='inner')
    merged = merged.dropna().reset_index(drop=True)
    print(f'  Aligned: {len(merged):,} samples | {merged["symbol"].nunique()} symbols')
    return merged

print('Aligning val predictions ...');  val_aligned  = align_predictions(lstm_val, gru_val, tft_val)
print('Aligning test predictions ...'); test_aligned = align_predictions(lstm_test, gru_test, tft_test)

Aligning val predictions ...
  Aligned: 6,027 samples | 49 symbols
Aligning test predictions ...
  Aligned: 9,061 samples | 49 symbols


In [57]:
def extract_stack_features(aligned_df):
    feat_cols = [f'{mdl}_q{qi}' for mdl in ['lstm','gru','tft'] for qi in range(NUM_QUANTILES)]
    X = aligned_df[feat_cols].values.astype(np.float32)
    y = aligned_df['true'].values.astype(np.float32)
    base_px = aligned_df['base_px'].values.astype(np.float32)
    true_px = aligned_df['true_px'].values.astype(np.float32)
    return X, y, base_px, true_px

X_val, y_val, bp_val, tp_val = extract_stack_features(val_aligned)
X_test, y_test, bp_test, tp_test = extract_stack_features(test_aligned)
print(f'Val  stacked features: {X_val.shape}')
print(f'Test stacked features: {X_test.shape}')

Val  stacked features: (6027, 21)
Test stacked features: (9061, 21)


## 8. Train ALL Meta-Learners & Auto-Select Best

In [58]:
import copy
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SKScaler

# MLP definition for 2 hidden layers
class StackingMLP2(nn.Module):
    def __init__(self, input_dim=21, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,32), nn.GELU(), # Last hidden layer has GELU but not BatchNorm1d or Dropout
            nn.Linear(32,1))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x).squeeze(-1)

# MLP definition for 3 hidden layers (original StackingMLP, renamed for clarity)
class StackingMLP3(nn.Module):
    def __init__(self, input_dim=21, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,32), nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(32,16), nn.GELU(), # Last hidden layer has GELU but not BatchNorm1d or Dropout
            nn.Linear(16,1))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x).squeeze(-1)

# MLP definition for 4 hidden layers
class StackingMLP4(nn.Module):
    def __init__(self, input_dim=21, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,32), nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(32,16), nn.BatchNorm1d(16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16,8), nn.GELU(), # Last hidden layer has GELU but not BatchNorm1d or Dropout
            nn.Linear(8,1))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x).squeeze(-1)

# MLP definition for 5 hidden layers
class StackingMLP5(nn.Module):
    def __init__(self, input_dim=21, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,32), nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(32,16), nn.BatchNorm1d(16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16,8), nn.BatchNorm1d(8), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(8,4), nn.GELU(), # Last hidden layer has GELU but not BatchNorm1d or Dropout
            nn.Linear(4,1))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp(mlp_class, X_tr, y_tr, X_mv, y_mv, input_dim, epochs=100, patience=15, lr=1e-3, bs=512):
    model = mlp_class(input_dim=input_dim).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=0.5,patience=5,min_lr=1e-6)
    loss_fn = nn.MSELoss()
    Xt=torch.tensor(X_tr,device=DEVICE); yt=torch.tensor(y_tr,device=DEVICE)
    dl = DataLoader(TensorDataset(Xt,yt), batch_size=bs, shuffle=True, drop_last=True)
    best_loss, best_state, pat, history = float('inf'), None, 0, []
    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in dl:
            opt.zero_grad(); l=loss_fn(model(xb),yb); l.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=loss_fn(model(torch.tensor(X_mv, device=DEVICE)),torch.tensor(y_mv, device=DEVICE)).item()
        sched.step(vl); history.append(vl)
        if vl < best_loss - 1e-7:
            best_loss=vl; pat=0; best_state={k:v.clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best_loss, history

# Helper function to predict for a given model (sklearn or torch)
def _meta_predict(model_info, X):
    if model_info['type'] == 'sklearn':
        return model_info['model'].predict(X).astype(np.float32)
    else: # torch model
        m = model_info['model']
        m.eval()
        with torch.no_grad():
            return m(torch.tensor(X, device=DEVICE)).cpu().numpy()

# Split val into meta-train (70%) / meta-val (30%)
n_val = len(X_val); perm = np.random.permutation(n_val)
mt_idx, mv_idx = perm[:int(0.7*n_val)], perm[int(0.7*n_val):]
X_mt, y_mt = X_val[mt_idx], y_val[mt_idx]
X_mv, y_mv = X_val[mv_idx], y_val[mv_idx]

candidates = {}

print('Training Ridge ...')
ridge = Pipeline([('sc',SKScaler()),('m',Ridge(alpha=1.0))]); ridge.fit(X_mt, y_mt)
ridge_preds_mv = _meta_predict({'model': ridge, 'type': 'sklearn'}, X_mv)
ridge_val_mse = float(np.mean((ridge_preds_mv - y_mv)**2))
ridge_val_metrics = compute_metrics(y_mv, ridge_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Ridge (L2)'] = {'model':ridge,'type':'sklearn','val_mse':ridge_val_mse, 'val_directional_acc': ridge_val_metrics['Directional_Acc']}
print(f'  val MSE = {ridge_val_mse:.6f} | Dir Acc = {ridge_val_metrics["Directional_Acc"]:.4f}')

print('Training ElasticNet ...')
enet = Pipeline([('sc',SKScaler()),('m',ElasticNet(alpha=0.001,l1_ratio=0.5,max_iter=5000))]); enet.fit(X_mt,y_mt)
enet_preds_mv = _meta_predict({'model': enet, 'type': 'sklearn'}, X_mv)
enet_val_mse = float(np.mean((enet_preds_mv - y_mv)**2))
enet_val_metrics = compute_metrics(y_mv, enet_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['ElasticNet'] = {'model':enet,'type':'sklearn','val_mse':enet_val_mse, 'val_directional_acc': enet_val_metrics['Directional_Acc']}
print(f'  val MSE = {enet_val_mse:.6f} | Dir Acc = {enet_val_metrics["Directional_Acc"]:.4f}')

print('Training Gradient Boosting ...')
gbr = GradientBoostingRegressor(n_estimators=200,max_depth=3,learning_rate=0.05,subsample=0.8,min_samples_leaf=20,random_state=SEED)
gbr.fit(X_mt,y_mt)
gbr_preds_mv = _meta_predict({'model': gbr, 'type': 'sklearn'}, X_mv)
gbr_val_mse = float(np.mean((gbr_preds_mv - y_mv)**2))
gbr_val_metrics = compute_metrics(y_mv, gbr_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Gradient Boosting'] = {'model':gbr,'type':'sklearn','val_mse':gbr_val_mse, 'val_directional_acc': gbr_val_metrics['Directional_Acc']}
print(f'  val MSE = {gbr_val_mse:.6f} | Dir Acc = {gbr_val_metrics["Directional_Acc"]:.4f}')

print('Training Random Forest ...')
rf = RandomForestRegressor(n_estimators=300,max_depth=5,min_samples_leaf=20,random_state=SEED,n_jobs=-1)
rf.fit(X_mt,y_mt)
rf_preds_mv = _meta_predict({'model': rf, 'type': 'sklearn'}, X_mv)
rf_val_mse = float(np.mean((rf_preds_mv - y_mv)**2))
rf_val_metrics = compute_metrics(y_mv, rf_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Random Forest'] = {'model':rf,'type':'sklearn','val_mse':rf_val_mse, 'val_directional_acc': rf_val_metrics['Directional_Acc']}
print(f'  val MSE = {rf_val_mse:.6f} | Dir Acc = {rf_val_metrics["Directional_Acc"]:.4f}')

print('Training SVR ...')
svr = Pipeline([('sc',SKScaler()),('m',SVR(kernel='rbf',C=1.0,epsilon=0.001))]); svr.fit(X_mt,y_mt)
svr_preds_mv = _meta_predict({'model': svr, 'type': 'sklearn'}, X_mv)
svr_val_mse = float(np.mean((svr_preds_mv - y_mv)**2))
svr_val_metrics = compute_metrics(y_mv, svr_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['SVR (RBF)'] = {'model':svr,'type':'sklearn','val_mse':svr_val_mse, 'val_directional_acc': svr_val_metrics['Directional_Acc']}
print(f'  val MSE = {svr_val_mse:.6f} | Dir Acc = {svr_val_metrics["Directional_Acc"]:.4f}')

print('Training StackingMLP2 (2 layers) ...')
mlp2_model, mlp2_val_mse, mlp2_history = train_mlp(StackingMLP2, X_mt, y_mt, X_mv, y_mv, input_dim=X_val.shape[1])
mlp2_preds_mv = _meta_predict({'model': mlp2_model, 'type': 'torch'}, X_mv)
mlp2_val_metrics = compute_metrics(y_mv, mlp2_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Stacking MLP2'] = {'model':mlp2_model,'type':'torch','val_mse':mlp2_val_mse,'history':mlp2_history, 'val_directional_acc': mlp2_val_metrics['Directional_Acc']}
print(f'  val MSE = {mlp2_val_mse:.6f} | Dir Acc = {mlp2_val_metrics["Directional_Acc"]:.4f}')

print('Training StackingMLP3 (3 layers) ...')
mlp3_model, mlp3_val_mse, mlp3_history = train_mlp(StackingMLP3, X_mt, y_mt, X_mv, y_mv, input_dim=X_val.shape[1])
mlp3_preds_mv = _meta_predict({'model': mlp3_model, 'type': 'torch'}, X_mv)
mlp3_val_metrics = compute_metrics(y_mv, mlp3_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Stacking MLP3'] = {'model':mlp3_model,'type':'torch','val_mse':mlp3_val_mse,'history':mlp3_history, 'val_directional_acc': mlp3_val_metrics['Directional_Acc']}
print(f'  val MSE = {mlp3_val_mse:.6f} | Dir Acc = {mlp3_val_metrics["Directional_Acc"]:.4f}')

print('Training StackingMLP4 (4 layers) ...')
mlp4_model, mlp4_val_mse, mlp4_history = train_mlp(StackingMLP4, X_mt, y_mt, X_mv, y_mv, input_dim=X_val.shape[1])
mlp4_preds_mv = _meta_predict({'model': mlp4_model, 'type': 'torch'}, X_mv)
mlp4_val_metrics = compute_metrics(y_mv, mlp4_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Stacking MLP4'] = {'model':mlp4_model,'type':'torch','val_mse':mlp4_val_mse,'history':mlp4_history, 'val_directional_acc': mlp4_val_metrics['Directional_Acc']}
print(f'  val MSE = {mlp4_val_mse:.6f} | Dir Acc = {mlp4_val_metrics["Directional_Acc"]:.4f}')

print('Training StackingMLP5 (5 layers) ...')
mlp5_model, mlp5_val_mse, mlp5_history = train_mlp(StackingMLP5, X_mt, y_mt, X_mv, y_mv, input_dim=X_val.shape[1])
mlp5_preds_mv = _meta_predict({'model': mlp5_model, 'type': 'torch'}, X_mv)
mlp5_val_metrics = compute_metrics(y_mv, mlp5_preds_mv, bp_val[mv_idx], tp_val[mv_idx])
candidates['Stacking MLP5'] = {'model':mlp5_model,'type':'torch','val_mse':mlp5_val_mse,'history':mlp5_history, 'val_directional_acc': mlp5_val_metrics['Directional_Acc']}
print(f'  val MSE = {mlp5_val_mse:.6f} | Dir Acc = {mlp5_val_metrics["Directional_Acc"]:.4f}')


# Leaderboard
print('\n' + '=' * 60)
print(' META-LEARNER VAL METRICS LEADERBOARD (Sorted by MSE)')
print('=' * 60)
sorted_cands_mse = sorted(candidates.items(), key=lambda x: x[1]['val_mse'])
for rank, (name, info) in enumerate(sorted_cands_mse, 1):
    marker = '  <-- BEST (MSE)' if rank == 1 else ''
    print(f'  {rank}. {name:22s}: MSE={info["val_mse"]:.6f} | Dir Acc={info["val_directional_acc"]:.4f}{marker}')

print('\n' + '=' * 60)
print(' META-LEARNER VAL METRICS LEADERBOARD (Sorted by Directional Accuracy)')
print('=' * 60)
sorted_cands_dir_acc = sorted(candidates.items(), key=lambda x: x[1]['val_directional_acc'], reverse=True)
for rank, (name, info) in enumerate(sorted_cands_dir_acc, 1):
    marker = '  <-- BEST (Dir Acc)' if rank == 1 else ''
    print(f'  {rank}. {name:22s}: Dir Acc={info["val_directional_acc"]:.4f} | MSE={info["val_mse"]:.6f}{marker}')


# Auto-select winner based on MSE as default
BEST_NAME = sorted_cands_mse[0][0]; BEST_INFO = sorted_cands_mse[0][1]
best_model = BEST_INFO['model']
print(f'\nSelected for FINAL ENSEMBLE (by default: best Val MSE): [{BEST_NAME}] (val MSE = {BEST_INFO["val_mse"]:.6f}, Dir Acc = {BEST_INFO["val_directional_acc"]:.4f})')

# Get history for plotting only if the best model is an MLP
history = None
if BEST_NAME.startswith('Stacking MLP'):
    history = BEST_INFO.get('history')
else:
    # Fallback to the history of the original StackingMLP3 if it exists, for plotting in cell_viz
    history = candidates.get('Stacking MLP3', {}).get('history')

Training Ridge ...
  val MSE = 0.000288 | Dir Acc = 0.5235
Training ElasticNet ...
  val MSE = 0.000283 | Dir Acc = 0.4942
Training Gradient Boosting ...
  val MSE = 0.000283 | Dir Acc = 0.5428
Training Random Forest ...
  val MSE = 0.000282 | Dir Acc = 0.5290
Training SVR ...
  val MSE = 0.000302 | Dir Acc = 0.5755
Training StackingMLP2 (2 layers) ...
  val MSE = 0.000286 | Dir Acc = 0.5003
Training StackingMLP3 (3 layers) ...
  val MSE = 0.000285 | Dir Acc = 0.4870
Training StackingMLP4 (4 layers) ...
  val MSE = 0.000669 | Dir Acc = 0.4981
Training StackingMLP5 (5 layers) ...
  val MSE = 0.000299 | Dir Acc = 0.5202

 META-LEARNER VAL METRICS LEADERBOARD (Sorted by MSE)
  1. Random Forest         : MSE=0.000282 | Dir Acc=0.5290  <-- BEST (MSE)
  2. Gradient Boosting     : MSE=0.000283 | Dir Acc=0.5428
  3. ElasticNet            : MSE=0.000283 | Dir Acc=0.4942
  4. Stacking MLP3         : MSE=0.000285 | Dir Acc=0.4870
  5. Stacking MLP2         : MSE=0.000286 | Dir Acc=0.5003
  6. Rid

## 9. Generate Final Predictions & Full Comparison

In [59]:
def predict(model_info, X):
    if model_info['type'] == 'sklearn': return model_info['model'].predict(X).astype(np.float32)
    else:
        m = model_info['model']; m.eval()
        with torch.no_grad(): return m(torch.tensor(X, device=DEVICE)).cpu().numpy()

lstm_med = test_aligned['lstm_median'].values
gru_med  = test_aligned['gru_median'].values
tft_med  = test_aligned['tft_median'].values
simple_avg = (lstm_med + gru_med + tft_med) / 3.0

# Inverse-variance weights
val_mse_lstm = np.mean((val_aligned['lstm_median'].values-y_val)**2)
val_mse_gru  = np.mean((val_aligned['gru_median'].values -y_val)**2)
val_mse_tft  = np.mean((val_aligned['tft_median'].values -y_val)**2)
inv_var = np.array([1/val_mse_lstm, 1/val_mse_gru, 1/val_mse_tft])
ivw_w = inv_var / inv_var.sum()
ivw_preds = lstm_med*ivw_w[0] + gru_med*ivw_w[1] + tft_med*ivw_w[2]
print(f'Inv-Variance Weights: LSTM={ivw_w[0]:.3f}, GRU={ivw_w[1]:.3f}, TFT={ivw_w[2]:.3f}')

all_results = {}
for name, pred in [('LSTM (w30)',lstm_med),('GRU (w30)',gru_med),('TFT (w15)',tft_med),
                   ('Simple Average',simple_avg),('Inv-Var Blend',ivw_preds)]:
    all_results[name] = compute_metrics(y_test, pred, bp_test, tp_test)

for name, info in candidates.items():
    preds = predict(info, X_test)
    all_results[name] = compute_metrics(y_test, preds, bp_test, tp_test)
    all_results[name]['_preds'] = preds

best_preds = predict(BEST_INFO, X_test)

comp_df = pd.DataFrame({k:{kk:vv for kk,vv in v.items() if not kk.startswith('_')} for k,v in all_results.items()}).T
comp_df = comp_df[[c for c in REPORT_COLS if c in comp_df.columns]].astype(float).round(4)
print('\n' + '=' * 70)
print(f' FULL COMPARISON  |  Best Ensemble: [{BEST_NAME}]')
print('=' * 70)
display(HTML(comp_df.to_html()))
comp_df.to_csv(OUTPUT_DIR / 'results' / 'full_ensemble_comparison.csv')
print(f'Saved -> {OUTPUT_DIR / "results" / "full_ensemble_comparison.csv"}')

Inv-Variance Weights: LSTM=0.336, GRU=0.334, TFT=0.331

 FULL COMPARISON  |  Best Ensemble: [Random Forest]


,MAPE_price,MSE_pct,RMSE_pct,MAE_pct,F1_macro,Accuracy,Precision_macro,Recall_macro,Directional_Acc,n_samples
LSTM (w30),16.6000,2.2018,1.4838,1.0471,0.4517,0.4911,0.5010,0.5007,0.4911,9061.0
GRU (w30),16.5831,2.2034,1.4844,1.0464,0.4288,0.4992,0.4782,0.4879,0.4992,9061.0
TFT (w15),16.5714,2.2048,1.4849,1.0485,0.4122,0.5150,0.5041,0.5015,0.5150,9061.0
Simple Average,16.5842,2.1993,1.4830,1.0460,0.4419,0.5025,0.4868,0.4920,0.5025,9061.0
Inv-Var Blend,16.5843,2.1993,1.4830,1.0459,0.4427,0.5028,0.4875,0.4923,0.5028,9061.0
Ridge (L2),16.6324,2.2641,1.5047,1.0641,0.4367,0.4886,0.4989,0.4994,0.4886,9061.0
ElasticNet,16.6100,2.2109,1.4869,1.0510,0.3449,0.4837,0.4976,0.4998,0.4837,9061.0
Gradient Boosting,16.6670,2.3925,1.5468,1.1179,0.4769,0.4973,0.5055,0.5045,0.4973,9061.0
Random Forest,16.6208,2.2254,1.4918,1.0576,0.4486,0.4969,0.5121,0.5074,0.4969,9061.0
SVR (RBF),16.6499,2.8918,1.7005,1.1959,0.5036,0.5069,0.5106,0.5102,0.5069,9061.0


Saved -> ensemble_outputs/results/full_ensemble_comparison.csv


## 10. Visualisations

In [60]:
# 1. Val MSE Leaderboard
fig, ax = plt.subplots(figsize=(9, 4))
names = [n for n,_ in sorted_cands_mse]; mses = [i['val_mse'] for _,i in sorted_cands_mse]
colors = ['#2ecc71' if n==BEST_NAME else '#3498db' for n in names]
bars = ax.barh(range(len(names)), mses, color=colors, alpha=0.85)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
ax.invert_yaxis(); ax.set_xlabel('Val MSE (lower = better)')
ax.set_title('Meta-Learner Selection Leaderboard', fontweight='bold')
for j,v in enumerate(mses): ax.text(v, j, f'  {v:.5f}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'plots'/'meta_leaderboard.png',dpi=150,bbox_inches='tight'); plt.show()

# 2. Full metric bar chart (2x2)
metrics_to_plot = ['MAPE_price','MAE_pct','F1_macro','Directional_Acc']
model_names = list(all_results.keys()); cmap = plt.cm.plasma(np.linspace(0.1,0.9,len(model_names)))
fig, axes = plt.subplots(2,2,figsize=(16,11))
for i, metric in enumerate(metrics_to_plot):
    ax = axes[i//2, i%2]; vals = [all_results[n].get(metric,0) for n in model_names]
    bars = ax.barh(range(len(model_names)), vals, color=cmap, alpha=0.85)
    ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names, fontsize=7)
    ax.set_title(metric, fontsize=10, fontweight='bold'); ax.invert_yaxis(); ax.grid(True,alpha=0.3,axis='x')
    best_idx = int(np.argmin(vals)) if metric in ['MAPE_price','MAE_pct'] else int(np.argmax(vals))
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(2.5)
    for j,v in enumerate(vals): ax.text(v, j, f'  {v:.4f}', va='center', fontsize=6)
plt.suptitle(f'All Methods  |  Best: {BEST_NAME}', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'plots'/'full_comparison.png',dpi=150,bbox_inches='tight'); plt.show()

# 3. Best ensemble scatter
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(y_test, best_preds, alpha=0.1, s=5, c='steelblue')
lo,hi = min(y_test.min(),best_preds.min()), max(y_test.max(),best_preds.max())
ax.plot([lo,hi],[lo,hi],'r--',alpha=0.5); ax.set_xlabel('Actual Return'); ax.set_ylabel('Predicted Return')
ax.set_title(f'Best Ensemble [{BEST_NAME}] Actual vs Predicted', fontweight='bold'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'plots'/'best_scatter.png',dpi=150); plt.show()

# 4. MLP training curve
if history is not None:
    fig, ax = plt.subplots(figsize=(8,3)); ax.plot(history, label='MLP Val MSE', color='#9b59b6')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val MSE'); ax.set_title('MLP Training Curve')
    ax.legend(); ax.grid(True,alpha=0.3); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'plots'/'mlp_curve.png',dpi=150); plt.show()

## 11. Final Summary

In [61]:
m = all_results[BEST_NAME]
print('=' * 55)
print(f' FINAL ENSEMBLE: {BEST_NAME}')
print('=' * 55)
print(f'  Val MSE (selection criterion): {BEST_INFO["val_mse"]:.6f}')
print(f'  Val Directional Acc: {BEST_INFO["val_directional_acc"]:.4f}')
print(f'  Test MAPE_price  : {m.get("MAPE_price", 0):.4f} %')
print(f'  Test MAE_pct     : {m.get("MAE_pct", 0):.4f} pp')
print(f'  Test F1_macro    : {m.get("F1_macro", 0):.4f}')
print(f'  Test Dir Acc     : {m.get("Directional_Acc", 0):.4f}')
print(f'  Test n_samples   : {m.get("n_samples", 0):,.0f}')
print('=' * 55)
print(f'All outputs saved to: {OUTPUT_DIR}')

 FINAL ENSEMBLE: Random Forest
  Val MSE (selection criterion): 0.000282
  Val Directional Acc: 0.5290
  Test MAPE_price  : 16.6208 %
  Test MAE_pct     : 1.0576 pp
  Test F1_macro    : 0.4486
  Test Dir Acc     : 0.4969
  Test n_samples   : 9,061
All outputs saved to: ensemble_outputs


To download the `ensemble_outputs` directory, you can first compress it into a zip file and then download the compressed file. Run the following cells:

In [62]:
import shutil

# Zip the ensemble_outputs directory
shutil.make_archive('ensemble_outputs', 'zip', 'ensemble_outputs')
print('ensemble_outputs.zip created.')

ensemble_outputs.zip created.


In [63]:
from google.colab import files

# Download the zip file
files.download('ensemble_outputs.zip')
print('Downloading ensemble_outputs.zip...')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>